# 02 — Fine-Tuning via DeepSeek API

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 04_Fine_Tuning  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Upload de dataset** — enviar os arquivos JSONL para a DeepSeek API
- **Criar e monitorar um job** — submeter o fine-tuning e acompanhar o progresso
- **Usar o modelo fine-tunado** — chamar o modelo resultante como qualquer outro LLM
- **Avaliar a diferença** — comparar respostas do modelo base vs fine-tunado nas mesmas perguntas

---

### O que é fine-tuning?

O modelo base (`deepseek-chat`) foi treinado em dados gerais da internet.  
Fine-tuning é um treinamento adicional sobre um dataset específico — no nosso caso,
pares Q&A sobre o curso Especialista em IA.

O resultado é um modelo que:
- Conhece o conteúdo específico do curso com mais profundidade
- Responde no estilo e vocabulário do material do curso
- Precisa de menos tokens de contexto para responder bem (RAG opcional)

```
train.jsonl + val.jsonl
        │
        ▼
  [Upload API]  →  file_id
        │
        ▼
  [Criar Job]   →  job_id
        │
        ▼
  [Monitorar]   →  status: running → succeeded
        │
        ▼
  [Modelo FT]   →  fine_tuned_model id
        │
        ▼
  [Avaliação]   →  base vs fine-tunado
```

## Setup

In [ ]:
import sys, os, json, time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL = os.getenv('LLM_MODEL', 'deepseek-chat')
DATA_DIR  = Path('../data/finetune')

TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH   = DATA_DIR / 'val.jsonl'
IDS_PATH   = DATA_DIR / 'job_ids.json'   # persiste file_ids e job_id entre sessões

# Verifica que os arquivos do notebook anterior existem
for p in [TRAIN_PATH, VAL_PATH]:
    status = '✅' if p.exists() else '❌ NÃO ENCONTRADO'
    print(f'{status}  {p}')

print(f'\nLLM base : {LLM_MODEL}')

---
## 1. Upload dos Arquivos

Cada arquivo JSONL é enviado para a API e recebe um `file_id`.  
Os IDs são salvos localmente para não precisar fazer upload novamente se a sessão cair.

In [ ]:
def carregar_ids() -> dict:
    """Carrega file_ids e job_id salvos de sessões anteriores."""
    if IDS_PATH.exists():
        with open(IDS_PATH) as f:
            return json.load(f)
    return {}


def salvar_ids(ids: dict):
    """Persiste file_ids e job_id em disco."""
    with open(IDS_PATH, 'w') as f:
        json.dump(ids, f, indent=2)


def upload_arquivo(caminho: Path, purpose: str = 'fine-tune') -> str:
    """
    Faz upload de um arquivo JSONL para a DeepSeek API.
    Retorna o file_id.
    """
    print(f'Enviando {caminho.name} ({caminho.stat().st_size / 1024:.1f} KB)...')
    with open(caminho, 'rb') as f:
        resposta = llm.files.create(file=f, purpose=purpose)
    print(f'  file_id: {resposta.id}')
    return resposta.id


# Carrega IDs salvos — evita upload duplicado
IDS = carregar_ids()

if 'train_file_id' not in IDS:
    IDS['train_file_id'] = upload_arquivo(TRAIN_PATH)
    salvar_ids(IDS)
else:
    print(f'train_file_id já existe: {IDS["train_file_id"]}')

if 'val_file_id' not in IDS:
    IDS['val_file_id'] = upload_arquivo(VAL_PATH)
    salvar_ids(IDS)
else:
    print(f'val_file_id já existe: {IDS["val_file_id"]}')

print(f'\nIDs prontos:')
print(f'  train_file_id : {IDS["train_file_id"]}')
print(f'  val_file_id   : {IDS["val_file_id"]}')

---
## 2. Criar o Job de Fine-Tuning

O job submete os arquivos para treinamento com os hiperparâmetros configurados.  
A API retorna um `job_id` que usamos para monitorar o progresso.

### Hiperparâmetros

| Parâmetro | Valor | Descrição |
|---|---|---|
| `n_epochs` | 3 | Passagens completas sobre o dataset — 3 é o padrão recomendado |
| `batch_size` | `auto` | DeepSeek ajusta automaticamente baseado no tamanho do dataset |
| `learning_rate_multiplier` | `auto` | Multiplicador sobre a taxa base do modelo |

In [ ]:
def criar_job(train_file_id: str, val_file_id: str) -> str:
    """
    Cria um job de fine-tuning na DeepSeek API.
    Retorna o job_id.
    """
    job = llm.fine_tuning.jobs.create(
        training_file   = train_file_id,
        validation_file = val_file_id,
        model           = LLM_MODEL,
        hyperparameters = {
            'n_epochs'                 : 3,
            'batch_size'               : 'auto',
            'learning_rate_multiplier' : 'auto',
        },
    )
    print(f'Job criado!')
    print(f'  job_id : {job.id}')
    print(f'  status : {job.status}')
    print(f'  model  : {job.model}')
    return job.id


if 'job_id' not in IDS:
    IDS['job_id'] = criar_job(IDS['train_file_id'], IDS['val_file_id'])
    salvar_ids(IDS)
else:
    print(f'job_id já existe: {IDS["job_id"]}')
    print('Para criar um novo job, remova a chave "job_id" do arquivo job_ids.json')

JOB_ID = IDS['job_id']
print(f'\nJob ativo: {JOB_ID}')

---
## 3. Monitorar o Job

Fine-tuning pode levar de alguns minutos a horas dependendo do tamanho do dataset.  
O loop abaixo consulta o status a cada 60 segundos e exibe eventos de progresso.

In [ ]:
def checar_status(job_id: str) -> dict:
    """Retorna status atual do job e o fine_tuned_model se concluído."""
    job = llm.fine_tuning.jobs.retrieve(job_id)
    return {
        'status'            : job.status,
        'fine_tuned_model'  : job.fine_tuned_model,
        'trained_tokens'    : job.trained_tokens,
        'error'             : job.error,
    }


def monitorar_job(job_id: str, intervalo: int = 60, timeout: int = 7200):
    """
    Monitora o job em loop até concluir ou atingir timeout.

    intervalo: segundos entre cada consulta (padrão: 60s)
    timeout  : segundos máximos de espera (padrão: 2h)
    """
    print(f'Monitorando job {job_id}...')
    print(f'Intervalo: {intervalo}s | Timeout: {timeout//60}min')
    print(f'{"-"*50}')

    inicio     = time.time()
    status_ant = None

    while True:
        info       = checar_status(job_id)
        status     = info['status']
        decorrido  = int(time.time() - inicio)
        timestamp  = time.strftime('%H:%M:%S')

        if status != status_ant:
            print(f'[{timestamp}] +{decorrido:4d}s  status: {status}')
            status_ant = status

        if status == 'succeeded':
            print(f'\n✅ Fine-tuning concluído!')
            print(f'   Modelo     : {info["fine_tuned_model"]}')
            print(f'   Tokens     : {info["trained_tokens"]:,}')
            print(f'   Tempo total: {decorrido//60}min {decorrido%60}s')
            # Salva o model_id para uso nas próximas células
            IDS['fine_tuned_model'] = info['fine_tuned_model']
            salvar_ids(IDS)
            return info['fine_tuned_model']

        if status in ('failed', 'cancelled'):
            print(f'\n❌ Job encerrado com status: {status}')
            if info['error']:
                print(f'   Erro: {info["error"]}')
            return None

        if decorrido > timeout:
            print(f'\n⚠️  Timeout atingido ({timeout//60}min). Verifique manualmente.')
            return None

        time.sleep(intervalo)


# Verifica se já temos o modelo (sessão anterior completou)
if 'fine_tuned_model' in IDS:
    print(f'Modelo já disponível: {IDS["fine_tuned_model"]}')
    FT_MODEL = IDS['fine_tuned_model']
else:
    FT_MODEL = monitorar_job(JOB_ID)

---
## 4. Usar o Modelo Fine-Tunado

O modelo fine-tunado é chamado exatamente como qualquer outro — só muda o `model` parameter.  
O `fine_tuned_model` ID retornado pela API substitui `deepseek-chat`.

In [ ]:
# Carrega o model ID se ainda não estiver na memória
if 'FT_MODEL' not in dir() or not FT_MODEL:
    IDS     = carregar_ids()
    FT_MODEL = IDS.get('fine_tuned_model')
    if not FT_MODEL:
        raise ValueError('fine_tuned_model não encontrado. Execute as células anteriores.')

print(f'Modelo base       : {LLM_MODEL}')
print(f'Modelo fine-tunado: {FT_MODEL}')


SYSTEM_ASSISTENTE = """\
Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.
Responde perguntas técnicas sobre os módulos EAI_01 a EAI_08, cobrindo:
matemática para IA, machine learning, deep learning, NLP, visão computacional,
IA generativa, MLOps e big data com PySpark.
Seja preciso, técnico e direto. Use exemplos de código quando relevante.
"""


def perguntar(pergunta: str, model: str) -> str:
    """Faz uma pergunta a um modelo específico e retorna a resposta."""
    resp = llm.chat.completions.create(
        model    = model,
        messages = [
            {'role': 'system', 'content': SYSTEM_ASSISTENTE},
            {'role': 'user',   'content': pergunta},
        ],
    )
    return resp.choices[0].message.content


print('\nTeste rápido do modelo fine-tunado:')
pergunta_teste = 'O que é o EAI_07 e quais submódulos ele cobre?'
print(f'\n👤 {pergunta_teste}')
print(f'\n🤖 {perguntar(pergunta_teste, FT_MODEL)}')

---
## 5. Avaliação: Base vs Fine-Tunado

Comparamos os dois modelos nas mesmas perguntas sobre o curso.  
Usamos perguntas **que não estavam no dataset de treino** — as do split de validação.

In [ ]:
# Carrega perguntas do split de validação
perguntas_val = []
with open(VAL_PATH, 'r', encoding='utf-8') as f:
    for linha in f:
        ex  = json.loads(linha)
        msg = ex['messages']
        user_msg = next((m for m in msg if m['role'] == 'user'), None)
        ref_msg  = next((m for m in msg if m['role'] == 'assistant'), None)
        if user_msg and ref_msg:
            perguntas_val.append({
                'pergunta'  : user_msg['content'],
                'referencia': ref_msg['content'],
            })

print(f'Exemplos de validação carregados: {len(perguntas_val)}')

# Seleciona amostra aleatória para avaliação
import random
random.seed(42)
AMOSTRA_EVAL = random.sample(perguntas_val, min(10, len(perguntas_val)))
print(f'Amostra para avaliação: {len(AMOSTRA_EVAL)} perguntas')

In [ ]:
# Gera respostas dos dois modelos para a amostra
# ⚠️  Faz 2 × N chamadas à API — aguarda alguns segundos

print('Gerando respostas... (base + fine-tunado)\n')
resultados = []

for i, ex in enumerate(AMOSTRA_EVAL):
    print(f'  [{i+1}/{len(AMOSTRA_EVAL)}] {ex["pergunta"][:60]}...')

    resp_base = perguntar(ex['pergunta'], LLM_MODEL)
    time.sleep(0.5)
    resp_ft   = perguntar(ex['pergunta'], FT_MODEL)
    time.sleep(0.5)

    resultados.append({
        'pergunta'   : ex['pergunta'],
        'referencia' : ex['referencia'],
        'resp_base'  : resp_base,
        'resp_ft'    : resp_ft,
    })

print(f'\nRespostas geradas para {len(resultados)} perguntas.')

In [ ]:
# Avaliação por LLM-as-judge
# O próprio LLM avalia qual resposta é melhor em cada par

SYSTEM_JUIZ = """\
Você é um avaliador técnico imparcial. Compare duas respostas a uma pergunta sobre IA e ML.
Avalie precisão técnica, completude e clareza.
Responda APENAS com JSON válido, sem texto antes ou depois:
{"vencedor": "base" | "ft" | "empate", "justificativa": "<1 frase>"}
"""


def avaliar_par(pergunta: str, resp_base: str, resp_ft: str) -> dict:
    """Usa o LLM como juiz para comparar as duas respostas."""
    prompt = f"""Pergunta: {pergunta}

Resposta A (base):
{resp_base[:600]}

Resposta B (fine-tunado):
{resp_ft[:600]}

Qual resposta é melhor tecnicamente?"""

    try:
        resp = llm.chat.completions.create(
            model       = LLM_MODEL,
            messages    = [
                {'role': 'system', 'content': SYSTEM_JUIZ},
                {'role': 'user',   'content': prompt},
            ],
            temperature = 0.0,
        )
        texto = resp.choices[0].message.content.strip()
        import re
        texto = re.sub(r'^```json\s*|^```\s*|\s*```$', '', texto, flags=re.MULTILINE).strip()
        return json.loads(texto)
    except Exception as e:
        return {'vencedor': 'erro', 'justificativa': str(e)}


print('Avaliando pares com LLM-as-judge...\n')
avaliacoes = []

for i, r in enumerate(resultados):
    av = avaliar_par(r['pergunta'], r['resp_base'], r['resp_ft'])
    avaliacoes.append(av)
    print(f"  [{i+1}] {av['vencedor']:6} | {av['justificativa'][:80]}")
    time.sleep(0.3)

# Placar final
from collections import Counter
placar = Counter(a['vencedor'] for a in avaliacoes)
total  = len(avaliacoes)

print(f'\n{"-"*50}')
print(f'PLACAR FINAL ({total} perguntas):')
print(f'  Fine-tunado venceu : {placar["ft"]:2d} ({placar["ft"]/total*100:.0f}%)')
print(f'  Base venceu        : {placar["base"]:2d} ({placar["base"]/total*100:.0f}%)')
print(f'  Empate             : {placar["empate"]:2d} ({placar["empate"]/total*100:.0f}%)')

In [ ]:
# Exibe a comparação detalhada dos casos mais interessantes
# (onde o fine-tunado venceu claramente)

vitorias_ft = [
    (resultados[i], avaliacoes[i])
    for i in range(len(resultados))
    if avaliacoes[i]['vencedor'] == 'ft'
]

print(f'Exemplos onde fine-tunado foi melhor ({len(vitorias_ft)} casos):\n')

for r, av in vitorias_ft[:3]:   # mostra até 3
    print(f'{'='*60}')
    print(f'PERGUNTA : {r["pergunta"]}')
    print(f'JUIZ     : {av["justificativa"]}')
    print(f'\n── Base ──────────────────────────────────────────')
    print(r['resp_base'][:400])
    print(f'\n── Fine-Tunado ───────────────────────────────────')
    print(r['resp_ft'][:400])
    print()

---
## Resumo

| Etapa | Detalhe |
|---|---|
| **Upload** | `llm.files.create()` → `file_id` por arquivo |
| **Job** | `llm.fine_tuning.jobs.create()` → `job_id` |
| **Monitoramento** | `llm.fine_tuning.jobs.retrieve(job_id)` em loop |
| **Uso** | Mesmo cliente OpenAI, só muda o `model=fine_tuned_model` |
| **Avaliação** | LLM-as-judge compara base vs fine-tunado nas perguntas de validação |

### IDs persistidos

Todos os IDs são salvos em `data/finetune/job_ids.json` para retomar entre sessões:

```json
{
  "train_file_id"   : "file-...",
  "val_file_id"     : "file-...",
  "job_id"          : "ftjob-...",
  "fine_tuned_model": "ft:deepseek-chat:..."
}
```

### Observação sobre LLM-as-judge

Usar o modelo base como juiz pode ter viés — ele pode favorecer respostas no seu próprio estilo.  
Para uma avaliação mais robusta, use um modelo diferente como juiz ou avalie manualmente uma amostra.